In [ ]:
import gc
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import plotly.express as px

from data_processing import load_and_clean, vocab_sizes
from graph_construction import build_transaction_graph
from feature_engineering import engineer_all_features
from pyg_export import build_pyg_data, NUMERIC_FEATURE_COLUMNS
from neighbor_sampling import SimpleNeighborLoader
from model import LineMVGNN
from train import (
    train_model,
    run_inference,
    compute_classification_metrics,
    print_metrics_report,
)
from aggregation import build_transaction_view, aggregate_accounts

torch.manual_seed(0)
np.random.seed(0)

# ==========================================================
# Cache directory
# ==========================================================

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

VOCABS_PATH = CACHE_DIR / "vocabs.pkl"

TRAIN_ENGINEERED_PATH = CACHE_DIR / "df_train_engineered.parquet"
TEST_ENGINEERED_PATH = CACHE_DIR / "df_test_engineered.parquet"

SCALER_PATH = CACHE_DIR / "scaler.pkl"

CHECKPOINT_DIR = CACHE_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

In [ ]:
TRAIN_CSV = "HI-Small_Trans.csv"
TEST_CSV = "LI-Small_Trans.csv"

pd.read_csv(TRAIN_CSV, nrows=5)

In [ ]:
df_train, vocabs = load_and_clean(TRAIN_CSV)

if VOCABS_PATH.exists():

    print("Loading cached vocabulary...")

    with open(VOCABS_PATH, "rb") as f:
        vocabs = pickle.load(f)

else:

    print("Saving vocabulary...")

    with open(VOCABS_PATH, "wb") as f:
        pickle.dump(vocabs, f)

print(f"Train: {len(df_train):,} transactions, laundering rate {df_train['label'].mean():.3%}")
print(f"Memory (deep): {df_train.memory_usage(deep=True).sum() / 1e9:.3f} GB")

vocab_sizes(vocabs)

In [ ]:
g_train, src_train, dst_train, preds_train, succs_train = build_transaction_graph(df_train)

print(
    f"Train graph: {g_train.numberOfNodes():,} nodes, "
    f"{g_train.numberOfEdges():,} edges "
    f"(avg out-degree {g_train.numberOfEdges()/g_train.numberOfNodes():.2f})"
)

In [ ]:
if TRAIN_ENGINEERED_PATH.exists():

    print("Loading cached engineered training features...")

    df_train = pd.read_parquet(TRAIN_ENGINEERED_PATH)

else:

    print("Engineering training features...")

    df_train = engineer_all_features(
        df_train,
        preds_train,
        succs_train,
        src_train,
        dst_train,
    )

    df_train.to_parquet(TRAIN_ENGINEERED_PATH, index=False)

    print("Saved engineered training features.")

print(
    f"Engineered feature count: "
    f"{len(NUMERIC_FEATURE_COLUMNS)} numeric + "
    f"7 categorical embedding inputs"
)

df_train.filter(
    regex="^(sender_|receiver_|pair_|fan_|relay_|short_cycle)"
).head()

In [ ]:
data_train, scaler = build_pyg_data(
    df_train,
    src_train,
    dst_train,
    fit_scaler=True,
)

# Save scaler for later inference/testing
with open(SCALER_PATH, "wb") as f:
    pickle.dump(scaler, f)

print("Saved scaler.")
print(data_train)

In [ ]:
vs = vocab_sizes(vocabs)

model = LineMVGNN(
    numeric_dim=len(NUMERIC_FEATURE_COLUMNS),
    # n_accounts=vs["n_accounts"],
    # n_banks=vs["n_banks"],
    n_payment_formats=vs["n_payment_formats"],
    n_currencies=vs["n_currencies"],
    emb_dim=16,
    hidden_dim=64,
    num_layers=2,
    dropout=0.3,
)

print(model)

print(
    f"Trainable parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

In [ ]:
CHECKPOINT_PATH = CHECKPOINT_DIR / "latest_checkpoint.pt"

model, history = train_model(
    model, 
    data_train, 
    preds_train,
    num_epochs=8, 
    batch_size=512, 
    num_neighbors=(25, 15),
    lr=1e-3, 
    weight_decay=1e-4, 
    val_frac=0.15,
    neg_per_pos=70,
    checkpoint_path=CHECKPOINT_PATH,
    save_every_epoch=True,
    resume=True,
)

fig = px.line(
    pd.DataFrame(history).reset_index().rename(columns={"index": "epoch"}),
    x="epoch", y=["train_loss", "val_loss"],
    title="Training / validation loss curves",
    labels={"value": "BCE loss", "variable": "split"},
)
fig.show()

fig2 = px.line(
    pd.DataFrame(history).reset_index().rename(columns={"index": "epoch"}),
    x="epoch", y="val_pr_auc", title="Validation PR-AUC by epoch",
)
fig2.show()


In [ ]:
torch.save(
    model.state_dict(),
    "line_mvgnn_weights.pth",
)

print("Saved model weights.")

print("Vocabulary already cached.")

print("Scaler already cached.")

del (
    g_train,
    src_train,
    dst_train,
    preds_train,
    succs_train,
    df_train,
    data_train,
)

gc.collect()

print(
    "Freed training graph and feature objects."
)

In [ ]:
from sklearn.metrics import precision_recall_curve

# 1. Run inference on the VALIDATION set first (not test set)
val_idx = torch.arange(int(data_train.num_nodes * 0.85), data_train.num_nodes)
val_y = data_train.y[val_idx].numpy()

# You'll need to slice out validation probs (or re-run inference on val_idx)
# Assuming you have an array `val_probs` generated similarly to `test_probs`:
precisions, recalls, thresholds = precision_recall_curve(val_y, val_probs)

# 2. Find the threshold that maximizes the F1 score
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[best_idx]

print(f"Optimal Threshold (from Val set): {optimal_threshold:.4f}")
print(f"Expected Val F1: {f1_scores[best_idx]:.4f}")

# 3. Use THIS threshold for your test set metrics and dashboard
metrics = compute_classification_metrics(test_y, test_probs, threshold=optimal_threshold)
print_metrics_report(metrics)

In [ ]:
df_test, _ = load_and_clean(
    TEST_CSV,
    vocabs=vocabs,
)

print(
    f"Test: {len(df_test):,} transactions, "
    f"laundering rate "
    f"{df_test['label'].mean():.3%}"
)

print(
    f"Memory (deep): "
    f"{df_test.memory_usage(deep=True).sum()/1e9:.3f} GB"
)

In [ ]:
g_test, src_test, dst_test, preds_test, succs_test = \
    build_transaction_graph(df_test)

print(
    f"Test graph: "
    f"{g_test.numberOfNodes():,} nodes, "
    f"{g_test.numberOfEdges():,} edges"
)

if TEST_ENGINEERED_PATH.exists():

    print(
        "Loading cached engineered "
        "test features..."
    )

    df_test = pd.read_parquet(
        TEST_ENGINEERED_PATH
    )

else:

    print(
        "Engineering test features..."
    )

    df_test = engineer_all_features(
        df_test,
        preds_test,
        succs_test,
        src_test,
        dst_test,
    )

    df_test.to_parquet(
        TEST_ENGINEERED_PATH,
        index=False,
    )

    print(
        "Saved engineered test features."
    )

with open(
    SCALER_PATH,
    "rb",
) as f:

    scaler = pickle.load(f)

data_test, _ = build_pyg_data(
    df_test,
    src_test,
    dst_test,
    scaler=scaler,
    fit_scaler=False,
)

print(data_test)

In [ ]:
test_probs = run_inference(model, data_test, preds_test, batch_size=1024, num_neighbors=(25, 15))
test_y = data_test.y.numpy()

metrics = compute_classification_metrics(test_y, test_probs, threshold=0.5)
print_metrics_report(metrics)

cm = metrics["confusion_matrix"]
fig = px.imshow(
    cm, text_auto=True, color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=["Legitimate (0)", "Laundering (1)"], y=["Legitimate (0)", "Laundering (1)"],
    title="Confusion matrix - test set",
)
fig.show()


In [ ]:
reloaded = LineMVGNN(
    numeric_dim=len(NUMERIC_FEATURE_COLUMNS),
    n_accounts=vs["n_accounts"], n_banks=vs["n_banks"],
    n_payment_formats=vs["n_payment_formats"], n_currencies=vs["n_currencies"],
)
reloaded.load_state_dict(torch.load("line_mvgnn_weights.pth"))
reloaded.eval()
probs_reloaded = run_inference(reloaded, data_test, preds_test, batch_size=1024, num_neighbors=(25, 15))
print("Max abs difference vs. original model's probabilities:", np.abs(test_probs - probs_reloaded).max())


In [ ]:
RISK_THRESHOLD = 0.5

transaction_view = build_transaction_view(df_test, test_probs, threshold=RISK_THRESHOLD)
account_view = aggregate_accounts(df_test, test_probs, threshold=RISK_THRESHOLD)

transaction_view.to_csv("transaction_view.csv", index=False)
account_view.to_csv("account_view.csv", index=False)

print(f"{transaction_view['Flagged'].sum():,} / {len(transaction_view):,} transactions flagged "
      f"at threshold {RISK_THRESHOLD}")
print(f"{account_view['Account Alert'].sum():,} / {len(account_view):,} accounts alerted")
transaction_view.head(10)


In [ ]:
account_view.head(10)